# Fraud Detection ML Pipeline (CRISP-DM)

**Project**: Student fraud detection using `shop.db` (SQLite)

**Goal**: Predict whether an incoming order/payment is fraudulent (`orders.is_fraud`).

**Key constraints implemented**:
- **No target leakage**: we exclude labels/derived risk and any post-transaction data as predictors.
- **All preprocessing inside sklearn Pipelines**.
- **Frozen 80/20 test split** (stratified) with `random_state=27`, evaluated at the end.
- **Consistent CV object and scoring dict** across tuned models.

---

## 0) Setup


In [7]:
from __future__ import annotations

import json
import logging
import math
import os
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    StratifiedShuffleSplit,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 27

# Paths must not depend on cwd alone: Jupyter may start in `shop.db` or in `fraud_pipeline/`.
# Check `cwd/sql` first: if cwd is already `fraud_pipeline/`, that is the real project dir. A mistaken
# nested `fraud_pipeline/fraud_pipeline/sql` from an older run would wrongly match if checked first.
_cwd = Path.cwd().resolve()
if (_cwd / "sql").is_dir():
    PROJECT_DIR = _cwd
    BASE_DIR = PROJECT_DIR.parent
elif (_cwd / "fraud_pipeline" / "sql").is_dir():
    BASE_DIR = _cwd
    PROJECT_DIR = BASE_DIR / "fraud_pipeline"
else:
    raise FileNotFoundError(
        "Cannot locate fraud_pipeline SQL folder. "
        "Open or run the notebook with cwd = the shop.db project folder or fraud_pipeline/."
    )

DB_PATH = PROJECT_DIR / "shop.db"
MODELS_DIR = PROJECT_DIR / "models"
SQL_DIR = PROJECT_DIR / "sql"
REPORTS_DIR = PROJECT_DIR / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s - %(message)s",
)
logger = logging.getLogger("fraud_pipeline")

assert DB_PATH.exists(), f"Expected DB at {DB_PATH}"
logger.info("Using DB_PATH=%s", DB_PATH)

2026-04-01 20:49:48,691 INFO fraud_pipeline - Using DB_PATH=C:\Master Folder\IS 455 - Machine Learning\shop.db\fraud_pipeline\shop.db


## 1) Data understanding: inspect schema

In [8]:
def sqlite_connect(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn


def list_tables_and_columns(conn: sqlite3.Connection) -> pd.DataFrame:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
        conn,
    )
    rows: list[dict[str, Any]] = []
    for t in tables["name"].tolist():
        cols = pd.read_sql_query(f"PRAGMA table_info({t});", conn)
        for _, r in cols.iterrows():
            rows.append(
                {
                    "table": t,
                    "column": r["name"],
                    "type": r["type"],
                    "notnull": int(r["notnull"]),
                    "pk": int(r["pk"]),
                }
            )
    out = pd.DataFrame(rows).sort_values(["table", "pk", "column"]).reset_index(drop=True)
    return out


with sqlite_connect(DB_PATH) as conn:
    schema_df = list_tables_and_columns(conn)

schema_df

,table,column,type,notnull,pk
0,customers,birthdate,TEXT,1,0
1,customers,city,TEXT,0,0
2,customers,created_at,TEXT,1,0
3,customers,customer_segment,TEXT,0,0
4,customers,email,TEXT,1,0
5,customers,full_name,TEXT,1,0
6,customers,gender,TEXT,1,0
7,customers,is_active,INTEGER,1,0
8,customers,loyalty_tier,TEXT,0,0
9,customers,state,TEXT,0,0


### Prediction unit and target

- **Prediction unit**: one row per `orders.order_id`
- **Target**: `orders.is_fraud` (binary)
- **Hard-banned from predictors**: `orders.is_fraud`, `orders.risk_score`, and any post-transaction tables/columns

In [9]:
with sqlite_connect(DB_PATH) as conn:
    y_dist = pd.read_sql_query(
        """
        SELECT
          COUNT(*) AS n,
          SUM(is_fraud) AS n_fraud,
          AVG(is_fraud) AS fraud_rate
        FROM orders;
        """,
        conn,
    )

y_dist

,n,n_fraud,fraud_rate
0,5000,318,0.0636


## 2) Data preparation: build denormalized modeling dataset (1 row per order)

We build a leakage-safe dataset with one row per `orders.order_id`.

**Important**: we do *not* use `orders.is_fraud` or `orders.risk_score` as predictors, and we exclude post-transaction tables (`shipments`, `product_reviews`) from features.

In [10]:
def _safe_copy(df: pd.DataFrame) -> pd.DataFrame:
    return df.copy(deep=True)


def extract_tables_for_modeling(conn: sqlite3.Connection) -> dict[str, pd.DataFrame]:
    """Extract only tables needed for leakage-safe modeling at order time."""
    orders = pd.read_sql_query("SELECT * FROM orders;", conn)
    customers = pd.read_sql_query("SELECT * FROM customers;", conn)
    order_items = pd.read_sql_query("SELECT * FROM order_items;", conn)
    products = pd.read_sql_query("SELECT * FROM products;", conn)
    return {
        "orders": orders,
        "customers": customers,
        "order_items": order_items,
        "products": products,
    }


def split_order_datetime(orders: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of orders with order_datetime split into order_date and order_time.

    Does not mutate the input dataframe.
    """
    df = _safe_copy(orders)
    dt = pd.to_datetime(df["order_datetime"], errors="coerce", utc=False)
    df["order_date"] = dt.dt.date.astype("string")
    df["order_time"] = dt.dt.time.astype("string")
    return df


def build_order_item_aggregates(order_items: pd.DataFrame) -> pd.DataFrame:
    """Aggregate order_items to one row per order_id."""
    oi = _safe_copy(order_items)
    agg = (
        oi.groupby("order_id", as_index=False)
        .agg(
            n_lines=("order_item_id", "count"),
            sum_quantity=("quantity", "sum"),
            avg_unit_price=("unit_price", "mean"),
            min_unit_price=("unit_price", "min"),
            max_unit_price=("unit_price", "max"),
            items_total=("line_total", "sum"),
        )
        .reset_index(drop=True)
    )
    # Simple derived feature
    agg["n_items"] = agg["sum_quantity"]
    return agg


def build_product_mix_aggregates(order_items: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    """Aggregate product/category mix per order_id (safe at checkout)."""
    oi = _safe_copy(order_items)
    pr = _safe_copy(products)

    merged = oi.merge(pr[["product_id", "category", "price", "cost"]], on="product_id", how="left")
    merged["unit_margin"] = merged["price"] - merged["cost"]

    # Mode category per order
    def _mode(series: pd.Series) -> str | None:
        s = series.dropna()
        if s.empty:
            return None
        return s.value_counts().index[0]

    agg = (
        merged.groupby("order_id", as_index=False)
        .agg(
            n_unique_products=("product_id", pd.Series.nunique),
            n_unique_categories=("category", pd.Series.nunique),
            top_category=("category", _mode),
            avg_product_cost=("cost", "mean"),
            avg_unit_margin=("unit_margin", "mean"),
        )
        .reset_index(drop=True)
    )
    return agg


def build_modeling_dataset(
    orders: pd.DataFrame,
    customers: pd.DataFrame,
    order_items: pd.DataFrame,
    products: pd.DataFrame,
    include_customer_history: bool = True,
) -> pd.DataFrame:
    """Build denormalized dataset with one row per order.

    Returns a NEW dataframe; does not mutate inputs.

    include_customer_history:
      If True, add leakage-safe history features computed only from prior orders.
    """
    ord_df = split_order_datetime(orders)
    cust_df = _safe_copy(customers)

    oi_agg = build_order_item_aggregates(order_items)
    mix_agg = build_product_mix_aggregates(order_items, products)

    # Base join (one row per order)
    out = ord_df.merge(cust_df, on="customer_id", how="left", suffixes=("", "_customer"))
    out = out.merge(oi_agg, on="order_id", how="left")
    out = out.merge(mix_agg, on="order_id", how="left")

    # Discount proxy (still safe; derived from order-time totals)
    out["discount_proxy"] = out["order_subtotal"] - out["items_total"]

    if include_customer_history:
        # Build prior-history features using ONLY earlier orders per customer
        hist = out[["order_id", "customer_id", "order_datetime", "order_total"]].copy()
        hist["order_datetime_parsed"] = pd.to_datetime(hist["order_datetime"], errors="coerce", utc=False)
        hist = hist.sort_values(["customer_id", "order_datetime_parsed", "order_id"], kind="mergesort")

        # Prior count/spend (shifted so current row doesn't include itself)
        hist["customer_prior_orders"] = hist.groupby("customer_id").cumcount()
        hist["customer_prior_total_spend"] = (
            hist.groupby("customer_id")["order_total"].cumsum().shift(1).fillna(0.0)
        )

        # Days since previous order
        prev_dt = hist.groupby("customer_id")["order_datetime_parsed"].shift(1)
        hist["customer_days_since_prev_order"] = (
            (hist["order_datetime_parsed"] - prev_dt).dt.total_seconds() / 86400.0
        )

        out = out.merge(
            hist[["order_id", "customer_prior_orders", "customer_prior_total_spend", "customer_days_since_prev_order"]],
            on="order_id",
            how="left",
        )

    # Ensure one row per order
    if out["order_id"].duplicated().any():
        raise ValueError("Denormalization failed: duplicated order_id rows")

    return out


with sqlite_connect(DB_PATH) as conn:
    raw = extract_tables_for_modeling(conn)

model_df = build_modeling_dataset(**raw, include_customer_history=True)
model_df.head(5)

,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,...,n_items,n_unique_products,n_unique_categories,top_category,avg_product_cost,avg_unit_margin,discount_proxy,customer_prior_orders,customer_prior_total_spend,customer_days_since_prev_order
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,...,9,5,4,Beauty,41.120,28.122,0.000000e+00,907,397181.81,0.269421
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,...,7,5,3,Home,79.358,53.942,0.000000e+00,307,132392.79,0.104051
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,...,5,3,2,Garden,87.200,53.650,1.136868e-13,1013,440924.01,0.055995
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,...,1,1,1,Garden,82.660,54.940,0.000000e+00,764,338498.14,0.044282
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,...,1,1,1,Books,8.380,8.690,0.000000e+00,914,399275.21,0.081389


In [11]:
# Sanity checks
assert model_df.shape[0] == raw["orders"].shape[0], "Expected one row per order"

# Quick view of columns created
sorted([c for c in model_df.columns if c.startswith("order_")])[:20], model_df.shape

(['order_date',
  'order_datetime',
  'order_id',
  'order_subtotal',
  'order_time',
  'order_total'],
 (5000, 46))

## 3) Leakage audit (banned columns + post-transaction tables)

We enforce a conservative leakage policy:
- Ban targets/derived risk columns by name pattern.
- Ban any features sourced from post-transaction tables (e.g., `shipments`, `product_reviews`).
- Ensure only `orders.is_fraud` is used as the training label.

In [12]:
BANNED_NAME_PATTERNS = [
    "is_fraud",
    "fraud",
    "risk_score",
    "risk",
    "label",
    "target",
    "outcome",
    "review",
    "chargeback",
    "refund",
    "dispute",
]

BANNED_TABLES = {"shipments", "product_reviews"}  # post-transaction in this DB


def leakage_audit_columns(
    columns: Iterable[str],
    banned_name_patterns: Iterable[str] = BANNED_NAME_PATTERNS,
) -> list[str]:
    """Flag columns whose names suggest leakage/labels."""
    pats = [p.lower() for p in banned_name_patterns]
    banned: list[str] = []
    for c in columns:
        cl = c.lower()
        if any(p in cl for p in pats):
            banned.append(c)
    return sorted(set(banned))


def leakage_audit_sources(feature_sources: dict[str, str]) -> list[str]:
    """Flag features coming from known post-transaction tables.

    feature_sources maps feature_name -> source_table.
    """
    banned = [f for f, src in feature_sources.items() if src in BANNED_TABLES]
    return sorted(set(banned))


def enforce_no_leakage(
    df: pd.DataFrame,
    target_col: str,
    feature_sources: Optional[dict[str, str]] = None,
) -> dict[str, Any]:
    """Return a leakage report and the final feature column list.

    This function does not mutate df.
    """
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found")

    banned_by_name = leakage_audit_columns(df.columns)

    banned_by_source: list[str] = []
    if feature_sources is not None:
        banned_by_source = leakage_audit_sources(feature_sources)

    # Explicitly ban target and known label-like columns
    hard_ban = {target_col}
    if "risk_score" in df.columns:
        hard_ban.add("risk_score")

    banned_all = sorted(set(banned_by_name) | set(banned_by_source) | set(hard_ban))

    # Candidate features are everything except banned + identifiers we won't model directly
    non_feature_cols = {target_col, "order_id"}
    feature_cols = [c for c in df.columns if c not in set(banned_all) | non_feature_cols]

    report = {
        "target_col": target_col,
        "banned_by_name": banned_by_name,
        "banned_by_source": banned_by_source,
        "hard_ban": sorted(hard_ban),
        "banned_all": banned_all,
        "n_features_after_ban": len(feature_cols),
        "feature_cols": feature_cols,
    }
    return report


# For this notebook, we only used orders/customers/order_items/products in feature creation.
# Still, we track feature sources at a table-level for auditing/documentation.
FEATURE_SOURCES: dict[str, str] = {}
for c in model_df.columns:
    if c in raw["orders"].columns:
        FEATURE_SOURCES[c] = "orders"
    elif c in raw["customers"].columns:
        FEATURE_SOURCES[c] = "customers"
    elif c in {"n_lines", "sum_quantity", "avg_unit_price", "min_unit_price", "max_unit_price", "items_total", "n_items"}:
        FEATURE_SOURCES[c] = "order_items"
    elif c in {"n_unique_products", "n_unique_categories", "top_category", "avg_product_cost", "avg_unit_margin"}:
        FEATURE_SOURCES[c] = "products"
    else:
        # engineered from allowed sources
        FEATURE_SOURCES[c] = "engineered"

leakage_report = enforce_no_leakage(model_df, target_col="is_fraud", feature_sources=FEATURE_SOURCES)

# Show a compact audit summary
{
    "target_col": leakage_report["target_col"],
    "n_banned": len(leakage_report["banned_all"]),
    "banned_all": leakage_report["banned_all"],
    "n_features_after_ban": leakage_report["n_features_after_ban"],
}

{'target_col': 'is_fraud',
 'n_banned': 2,
 'banned_all': ['is_fraud', 'risk_score'],
 'n_features_after_ban': 43}

## 4) Frozen split (80/20) and consistent CV/scoring

We freeze a single stratified 80/20 split and persist indices so the test set is untouched until final evaluation.

In [13]:
SPLIT_PATH = MODELS_DIR / "split_indices.json"


def freeze_or_load_split(
    X: pd.DataFrame,
    y: pd.Series,
    split_path: Path,
    test_size: float = 0.2,
    random_state: int = RANDOM_STATE,
) -> dict[str, list[int]]:
    """Create or load a frozen stratified split.

    Stores row indices (positional indices) for train/test.
    """
    if split_path.exists():
        payload = json.loads(split_path.read_text(encoding="utf-8"))
        return {"train_idx": payload["train_idx"], "test_idx": payload["test_idx"]}

    splitter = StratifiedShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    (train_idx, test_idx) = next(splitter.split(X, y))

    out = {"train_idx": train_idx.tolist(), "test_idx": test_idx.tolist()}
    split_path.write_text(json.dumps(out, indent=2), encoding="utf-8")
    return out


TARGET_COL = "is_fraud"

feature_cols = leakage_report["feature_cols"]
X_all = model_df[feature_cols].copy()
y_all = model_df[TARGET_COL].astype(int).copy()

split = freeze_or_load_split(X_all, y_all, SPLIT_PATH)
train_idx = split["train_idx"]
test_idx = split["test_idx"]

X_train = X_all.iloc[train_idx].copy()
y_train = y_all.iloc[train_idx].copy()
X_test = X_all.iloc[test_idx].copy()
y_test = y_all.iloc[test_idx].copy()

logger.info("Frozen split saved at %s", SPLIT_PATH)
{
    "n_total": int(len(X_all)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "train_fraud_rate": float(y_train.mean()),
    "test_fraud_rate": float(y_test.mean()),
}

2026-04-01 20:49:50,936 INFO fraud_pipeline - Frozen split saved at C:\Master Folder\IS 455 - Machine Learning\shop.db\fraud_pipeline\models\split_indices.json


{'n_total': 5000,
 'n_train': 4000,
 'n_test': 1000,
 'train_fraud_rate': 0.0635,
 'test_fraud_rate': 0.064}

In [14]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING: dict[str, str] = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "neg_log_loss": "neg_log_loss",
}

SCORING

{'precision': 'precision',
 'recall': 'recall',
 'f1': 'f1',
 'average_precision': 'average_precision',
 'roc_auc': 'roc_auc',
 'neg_log_loss': 'neg_log_loss'}

## 5) Preprocessing + baseline models

All preprocessing happens inside a single sklearn `Pipeline` via a `ColumnTransformer`.

In [15]:
def build_preprocessor(X: pd.DataFrame) -> tuple[ColumnTransformer, list[str], list[str]]:
    """Create ColumnTransformer for numeric/categorical columns."""
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return pre, numeric_cols, categorical_cols


def cv_report(
    model_name: str,
    pipeline: Pipeline,
    X: pd.DataFrame,
    y: pd.Series,
    cv: StratifiedKFold,
    scoring: dict[str, str],
) -> pd.DataFrame:
    """Run cross_validate and return mean±std metrics as a dataframe."""
    res = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    rows: list[dict[str, Any]] = []
    for k, v in res.items():
        if k.startswith("test_"):
            metric = k.replace("test_", "")
            rows.append(
                {
                    "model": model_name,
                    "metric": metric,
                    "mean": float(np.mean(v)),
                    "std": float(np.std(v)),
                }
            )
        elif k == "fit_time":
            rows.append(
                {
                    "model": model_name,
                    "metric": "fit_time",
                    "mean": float(np.mean(v)),
                    "std": float(np.std(v)),
                }
            )

    return pd.DataFrame(rows).sort_values(["model", "metric"]).reset_index(drop=True)


preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train)
logger.info("Numeric cols: %d | Categorical cols: %d", len(numeric_cols), len(categorical_cols))
len(numeric_cols), len(categorical_cols)

2026-04-01 20:49:50,986 INFO fraud_pipeline - Numeric cols: 22 | Categorical cols: 21


(22, 21)

In [16]:
baseline_models: list[tuple[str, Any]] = [
    (
        "dummy_most_frequent",
        DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    ),
    (
        "logreg",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
    (
        "decision_tree",
        DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            max_depth=None,
        ),
    ),
]

cv_tables: list[pd.DataFrame] = []
for name, est in baseline_models:
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", est)])
    cv_tables.append(cv_report(name, pipe, X_train, y_train, CV, SCORING))

baseline_cv = pd.concat(cv_tables, ignore_index=True)
baseline_cv

,model,metric,mean,std
0,dummy_most_frequent,average_precision,0.063500,0.000500
1,dummy_most_frequent,f1,0.000000,0.000000
2,dummy_most_frequent,fit_time,0.084994,0.008135
3,dummy_most_frequent,neg_log_loss,-2.288772,0.018022
4,dummy_most_frequent,precision,0.000000,0.000000
5,dummy_most_frequent,recall,0.000000,0.000000
6,dummy_most_frequent,roc_auc,0.500000,0.000000
7,logreg,average_precision,0.154946,0.009467
8,logreg,f1,0.199812,0.030109
9,logreg,fit_time,0.460708,0.027315


In [17]:
def pivot_cv_table(cv_df: pd.DataFrame) -> pd.DataFrame:
    wide = cv_df.pivot_table(index="model", columns="metric", values=["mean", "std"])
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index().sort_values("model")
    return wide

pivot_cv_table(baseline_cv)

,model,mean_average_precision,mean_f1,mean_fit_time,mean_neg_log_loss,mean_precision,mean_recall,mean_roc_auc,std_average_precision,std_f1,std_fit_time,std_neg_log_loss,std_precision,std_recall,std_roc_auc
0,decision_tree,0.072836,0.133913,0.322198,-4.937981,0.111034,0.169412,0.539724,0.005533,0.034753,0.054729,0.177036,0.026111,0.049443,0.022004
1,dummy_most_frequent,0.063500,0.000000,0.084994,-2.288772,0.000000,0.000000,0.500000,0.000500,0.000000,0.008135,0.018022,0.000000,0.000000,0.000000
2,logreg,0.154946,0.199812,0.460708,-0.302872,0.176761,0.232471,0.707700,0.009467,0.030109,0.027315,0.014019,0.021778,0.049678,0.017921


## 6) Model tuning (shared CV + scoring)

We tune `RandomForestClassifier` and `GradientBoostingClassifier` using the same `CV` object and `SCORING` dict.

In [18]:
def tune_model(
    name: str,
    base_pipeline: Pipeline,
    param_grid: dict[str, list[Any]],
    X: pd.DataFrame,
    y: pd.Series,
    cv: StratifiedKFold,
    scoring: dict[str, str],
    refit_metric: str = "average_precision",
) -> GridSearchCV:
    """GridSearchCV wrapper with shared CV/scoring."""
    gs = GridSearchCV(
        estimator=base_pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit=refit_metric,
        cv=cv,
        n_jobs=-1,
        verbose=1,
        return_train_score=False,
    )
    gs.fit(X, y)
    logger.info("%s best_params=%s best_%s=%.4f", name, gs.best_params_, refit_metric, gs.best_score_)
    return gs


def cv_metrics_from_gridsearch(gs: GridSearchCV, model_name: str) -> pd.DataFrame:
    """Compute mean±std CV metrics from best estimator CV results."""
    i = int(gs.best_index_)
    rows: list[dict[str, Any]] = []
    for metric in SCORING.keys():
        mean_key = f"mean_test_{metric}"
        std_key = f"std_test_{metric}"
        if mean_key in gs.cv_results_:
            rows.append(
                {
                    "model": model_name,
                    "metric": metric,
                    "mean": float(gs.cv_results_[mean_key][i]),
                    "std": float(gs.cv_results_[std_key][i]),
                }
            )
    rows.append(
        {
            "model": model_name,
            "metric": "fit_time",
            "mean": float(gs.cv_results_["mean_fit_time"][i]),
            "std": float(gs.cv_results_["std_fit_time"][i]),
        }
    )
    return pd.DataFrame(rows).sort_values(["model", "metric"]).reset_index(drop=True)


rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
rf_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", rf)])
rf_grid = {
    "model__max_depth": [None, 6, 12],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt", 0.5],
}

rf_gs = tune_model("random_forest", rf_pipe, rf_grid, X_train, y_train, CV, SCORING, refit_metric="average_precision")
rf_cv = cv_metrics_from_gridsearch(rf_gs, "random_forest")


gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
gb_pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", gb)])
gb_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__subsample": [1.0, 0.8],
}

gb_gs = tune_model("gradient_boosting", gb_pipe, gb_grid, X_train, y_train, CV, SCORING, refit_metric="average_precision")
gb_cv = cv_metrics_from_gridsearch(gb_gs, "gradient_boosting")


tuned_cv = pd.concat([rf_cv, gb_cv], ignore_index=True)
pivot_cv_table(tuned_cv)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


2026-04-01 20:53:01,135 INFO fraud_pipeline - random_forest best_params={'model__max_depth': 6, 'model__max_features': 0.5, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2} best_average_precision=0.1656


Fitting 5 folds for each of 16 candidates, totalling 80 fits


2026-04-01 20:54:48,005 INFO fraud_pipeline - gradient_boosting best_params={'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 200, 'model__subsample': 1.0} best_average_precision=0.1714


,model,mean_average_precision,mean_f1,mean_fit_time,mean_neg_log_loss,mean_precision,mean_recall,mean_roc_auc,std_average_precision,std_f1,std_fit_time,std_neg_log_loss,std_precision,std_recall,std_roc_auc
0,gradient_boosting,0.171426,0.000000,19.999722,-0.216980,0.000000,0.000000,0.750447,0.038231,0.00000,1.184257,0.004813,0.000000,0.000000,0.025386
1,random_forest,0.165607,0.228834,12.406236,-0.402601,0.148301,0.507373,0.746457,0.023782,0.02735,1.696464,0.011974,0.018325,0.079283,0.019148


In [19]:
all_cv = pd.concat([baseline_cv, tuned_cv], ignore_index=True)
pivot_cv_table(all_cv)

,model,mean_average_precision,mean_f1,mean_fit_time,mean_neg_log_loss,mean_precision,mean_recall,mean_roc_auc,std_average_precision,std_f1,std_fit_time,std_neg_log_loss,std_precision,std_recall,std_roc_auc
0,decision_tree,0.072836,0.133913,0.322198,-4.937981,0.111034,0.169412,0.539724,0.005533,0.034753,0.054729,0.177036,0.026111,0.049443,0.022004
1,dummy_most_frequent,0.063500,0.000000,0.084994,-2.288772,0.000000,0.000000,0.500000,0.000500,0.000000,0.008135,0.018022,0.000000,0.000000,0.000000
2,gradient_boosting,0.171426,0.000000,19.999722,-0.216980,0.000000,0.000000,0.750447,0.038231,0.000000,1.184257,0.004813,0.000000,0.000000,0.025386
3,logreg,0.154946,0.199812,0.460708,-0.302872,0.176761,0.232471,0.707700,0.009467,0.030109,0.027315,0.014019,0.021778,0.049678,0.017921
4,random_forest,0.165607,0.228834,12.406236,-0.402601,0.148301,0.507373,0.746457,0.023782,0.027350,1.696464,0.011974,0.018325,0.079283,0.019148


## 7) Frozen test evaluation (top 2 models only, evaluated once)

We evaluate the two strongest CV models on the frozen test set **exactly once**.

In [20]:
def evaluate_on_test_once(
    fitted_pipeline: Pipeline,
    X_test: pd.DataFrame,
    y_test: pd.Series,
) -> dict[str, Any]:
    """Compute test metrics using predicted probabilities (threshold-free)."""
    proba = fitted_pipeline.predict_proba(X_test)[:, 1]
    metrics = {
        "average_precision": float(average_precision_score(y_test, proba)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "log_loss": float(log_loss(y_test, proba, labels=[0, 1])),
    }
    return metrics


# Choose the best two by CV average_precision (PR AUC)
cv_ap = (
    all_cv[all_cv["metric"] == "average_precision"]
    .sort_values(["mean"], ascending=False)
    .reset_index(drop=True)
)
cv_ap[["model", "mean", "std"]].head(10)

,model,mean,std
0,gradient_boosting,0.171426,0.038231
1,random_forest,0.165607,0.023782
2,logreg,0.154946,0.009467
3,decision_tree,0.072836,0.005533
4,dummy_most_frequent,0.063500,0.000500


In [21]:
top2 = cv_ap["model"].head(2).tolist()
logger.info("Top-2 CV by average_precision: %s", top2)

# Map names to fitted estimators (fit on training split only)
model_registry: dict[str, Any] = {
    "random_forest": rf_gs.best_estimator_,
    "gradient_boosting": gb_gs.best_estimator_,
    "logreg": Pipeline(steps=[("preprocess", preprocessor), ("model", baseline_models[1][1])]),
    "decision_tree": Pipeline(steps=[("preprocess", preprocessor), ("model", baseline_models[2][1])]),
    "dummy_most_frequent": Pipeline(steps=[("preprocess", preprocessor), ("model", baseline_models[0][1])]),
}

# Fit top-2 on train split (only)
fitted_top2: dict[str, Pipeline] = {}
for name in top2:
    pipe = model_registry[name]
    pipe.fit(X_train, y_train)
    fitted_top2[name] = pipe

# Evaluate once on frozen test
TEST_METRICS_PATH = MODELS_DIR / "test_metrics_once.json"
if TEST_METRICS_PATH.exists():
    test_metrics = json.loads(TEST_METRICS_PATH.read_text(encoding="utf-8"))
else:
    test_metrics = {name: evaluate_on_test_once(pipe, X_test, y_test) for name, pipe in fitted_top2.items()}
    TEST_METRICS_PATH.write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")

pd.DataFrame(
    [
        {"model": m, **met}
        for m, met in test_metrics.items()
    ]
).sort_values("average_precision", ascending=False)

2026-04-01 20:54:48,121 INFO fraud_pipeline - Top-2 CV by average_precision: ['gradient_boosting', 'random_forest']


,model,average_precision,roc_auc,log_loss
0,gradient_boosting,0.148741,0.732514,0.218614
1,random_forest,0.145283,0.739633,0.417395


## 8) Threshold selection (precision/recall tradeoff) + save final artifacts

We pick a threshold using the **train split only** to meet a recall target while maximizing precision (policy is documented in metadata).

In [22]:
def select_threshold_by_recall(
    y_true: pd.Series,
    proba: np.ndarray,
    recall_min: float = 0.80,
) -> dict[str, Any]:
    """Select smallest threshold achieving recall_min; among those, maximize precision."""
    precision, recall, thresholds = precision_recall_curve(y_true, proba)

    # precision_recall_curve returns thresholds of length n-1
    # Align arrays: thresholds correspond to precision[1:], recall[1:]
    precision_t = precision[1:]
    recall_t = recall[1:]

    ok = recall_t >= recall_min
    if not np.any(ok):
        # fallback: maximize F1
        f1 = 2 * (precision_t * recall_t) / np.clip(precision_t + recall_t, 1e-12, None)
        j = int(np.argmax(f1))
        return {
            "policy": "max_f1_fallback",
            "threshold": float(thresholds[j]),
            "precision": float(precision_t[j]),
            "recall": float(recall_t[j]),
            "recall_min": recall_min,
        }

    idx = np.where(ok)[0]
    # pick max precision among ok; tie-break by higher threshold
    best = idx[np.argmax(precision_t[idx])]

    return {
        "policy": "max_precision_given_recall_min",
        "threshold": float(thresholds[best]),
        "precision": float(precision_t[best]),
        "recall": float(recall_t[best]),
        "recall_min": recall_min,
    }


def risk_band_1_100(proba: np.ndarray) -> np.ndarray:
    band = np.ceil(100.0 * proba).astype(int)
    return np.clip(band, 1, 100)


# Choose final model based on test_metrics_once PR AUC
final_model_name = (
    pd.DataFrame([{"model": m, **met} for m, met in test_metrics.items()])
    .sort_values("average_precision", ascending=False)
    .iloc[0]["model"]
)
final_pipe = fitted_top2[final_model_name]
logger.info("Final model selected: %s", final_model_name)

# Threshold selection uses TRAIN predictions only
train_proba = final_pipe.predict_proba(X_train)[:, 1]
threshold_info = select_threshold_by_recall(y_train, train_proba, recall_min=0.80)
threshold_info

2026-04-01 20:55:02,588 INFO fraud_pipeline - Final model selected: gradient_boosting


{'policy': 'max_precision_given_recall_min',
 'threshold': 0.16127760559935622,
 'precision': 0.7788461538461539,
 'recall': 0.9566929133858267,
 'recall_min': 0.8}

In [23]:
def eval_at_threshold(y_true: pd.Series, proba: np.ndarray, threshold: float) -> dict[str, Any]:
    pred = (proba >= threshold).astype(int)
    return {
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(2 * precision_score(y_true, pred, zero_division=0) * recall_score(y_true, pred, zero_division=0) / max(precision_score(y_true, pred, zero_division=0) + recall_score(y_true, pred, zero_division=0), 1e-12)),
        "confusion_matrix": confusion_matrix(y_true, pred).tolist(),
    }


test_proba = final_pipe.predict_proba(X_test)[:, 1]
threshold = float(threshold_info["threshold"])

threshold_test_metrics = eval_at_threshold(y_test, test_proba, threshold)
threshold_test_metrics

{'precision': 0.21621621621621623,
 'recall': 0.125,
 'f1': 0.15841584158415842,
 'confusion_matrix': [[907, 29], [56, 8]]}

In [24]:
MODEL_PATH = MODELS_DIR / "model.joblib"
METADATA_PATH = MODELS_DIR / "metadata.json"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def build_metadata(
    model_name: str,
    feature_cols: list[str],
    leakage_report: dict[str, Any],
    cv_table: pd.DataFrame,
    test_metrics_once: dict[str, Any],
    threshold_info: dict[str, Any],
    threshold_test_metrics: dict[str, Any],
) -> dict[str, Any]:
    return {
        "model_version": "v1",
        "trained_at_utc": utc_now_iso(),
        "random_state": RANDOM_STATE,
        "db_path": str(DB_PATH),
        "prediction_unit": "orders.order_id",
        "target_col": TARGET_COL,
        "feature_cols_raw": feature_cols,
        "leakage": {
            "banned_name_patterns": BANNED_NAME_PATTERNS,
            "banned_tables": sorted(BANNED_TABLES),
            "audit": leakage_report,
        },
        "cv_metrics": (
            cv_table.sort_values(["model", "metric"]).to_dict(orient="records")
        ),
        "test_metrics_once": test_metrics_once,
        "final_model": {
            "name": model_name,
            "threshold": threshold_info,
            "threshold_test_metrics": threshold_test_metrics,
        },
        "environment": {
            "python": f"{os.sys.version_info.major}.{os.sys.version_info.minor}.{os.sys.version_info.micro}",
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }


# Save model pipeline
joblib.dump(final_pipe, MODEL_PATH)

metadata = build_metadata(
    model_name=final_model_name,
    feature_cols=feature_cols,
    leakage_report=leakage_report,
    cv_table=all_cv,
    test_metrics_once=test_metrics,
    threshold_info=threshold_info,
    threshold_test_metrics=threshold_test_metrics,
)
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

{"model_saved": str(MODEL_PATH), "metadata_saved": str(METADATA_PATH)}

{'model_saved': 'C:\\Master Folder\\IS 455 - Machine Learning\\shop.db\\fraud_pipeline\\models\\model.joblib',
 'metadata_saved': 'C:\\Master Folder\\IS 455 - Machine Learning\\shop.db\\fraud_pipeline\\models\\metadata.json'}

## 9) Inference + writeback demo (SQLite)

This section creates `payment_predictions` (if needed), scores orders, assigns risk bands, and writes results back to SQLite.

In [25]:
PAYMENT_PREDICTIONS_DDL = (SQL_DIR / "payment_predictions.sql").read_text(encoding="utf-8")


def ensure_payment_predictions_table(conn: sqlite3.Connection) -> None:
    conn.executescript(PAYMENT_PREDICTIONS_DDL)
    conn.commit()


def write_predictions(
    conn: sqlite3.Connection,
    df_pred: pd.DataFrame,
    model_version: str,
    threshold: float,
    metadata_json: str,
) -> None:
    ensure_payment_predictions_table(conn)

    payload = df_pred.copy()
    payload["model_version"] = model_version
    payload["scored_at_utc"] = utc_now_iso()
    payload["threshold"] = threshold
    payload["metadata_json"] = metadata_json

    rows = payload.to_dict(orient="records")
    conn.executemany(
        """
        INSERT INTO payment_predictions
          (order_id, model_version, scored_at_utc, proba_fraud, risk_band_1_100, threshold, is_fraud_pred, metadata_json)
        VALUES
          (:order_id, :model_version, :scored_at_utc, :proba_fraud, :risk_band_1_100, :threshold, :is_fraud_pred, :metadata_json)
        ON CONFLICT(order_id, model_version) DO UPDATE SET
          scored_at_utc=excluded.scored_at_utc,
          proba_fraud=excluded.proba_fraud,
          risk_band_1_100=excluded.risk_band_1_100,
          threshold=excluded.threshold,
          is_fraud_pred=excluded.is_fraud_pred,
          metadata_json=excluded.metadata_json;
        """,
        rows,
    )
    conn.commit()


MODEL_PATH = MODELS_DIR / "model.joblib"
METADATA_PATH = MODELS_DIR / "metadata.json"

# After a kernel restart (or if earlier training cells were skipped), reload saved artifacts.
if "final_pipe" not in globals():
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(
            f"{MODEL_PATH} not found. Run the training cells through model save, or Run All top-to-bottom."
        )
    final_pipe = joblib.load(MODEL_PATH)

if "metadata" not in globals():
    if not METADATA_PATH.is_file():
        raise FileNotFoundError(
            f"{METADATA_PATH} not found. Run the cell that writes metadata.json, or Run All top-to-bottom."
        )
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
if "final_model_name" not in globals():
    final_model_name = metadata["final_model"]["name"]
if "threshold" not in globals():
    threshold = float(metadata["final_model"]["threshold"]["threshold"])

if "risk_band_1_100" not in globals():

    def risk_band_1_100(proba: np.ndarray) -> np.ndarray:
        band = np.ceil(100.0 * proba).astype(int)
        return np.clip(band, 1, 100)


# Score ALL orders (demo). In a real app you would score only new/unscored orders.
all_proba = final_pipe.predict_proba(X_all)[:, 1]
all_pred = (all_proba >= threshold).astype(int)

pred_out = pd.DataFrame(
    {
        "order_id": model_df["order_id"].astype(int).values,
        "proba_fraud": all_proba.astype(float),
        "risk_band_1_100": risk_band_1_100(all_proba),
        "is_fraud_pred": all_pred.astype(int),
    }
)

with sqlite_connect(DB_PATH) as conn:
    write_predictions(
        conn,
        df_pred=pred_out,
        model_version=metadata["model_version"],
        threshold=threshold,
        metadata_json=json.dumps({"final_model": final_model_name}),
    )

pred_out.head(10)

,order_id,proba_fraud,risk_band_1_100,is_fraud_pred
0,1,0.127758,13,0
1,2,0.190029,20,1
2,3,0.191299,20,1
3,4,0.039110,4,0
4,5,0.039110,4,0
5,6,0.095155,10,0
6,7,0.134225,14,0
7,8,0.134225,14,0
8,9,0.055706,6,0
9,10,0.039110,4,0


In [26]:
# Optional: create the app view
VIEW_SQL_PATH = SQL_DIR / "risky_transactions_view.sql"
with sqlite_connect(DB_PATH) as conn:
    conn.executescript(VIEW_SQL_PATH.read_text(encoding="utf-8"))
    conn.commit()

# Preview risky transactions (top risk)
with sqlite_connect(DB_PATH) as conn:
    df_risky = pd.read_sql_query(
        """
        SELECT *
        FROM vw_risky_transactions
        ORDER BY proba_fraud DESC
        LIMIT 20;
        """,
        conn,
    )

df_risky

,order_id,order_datetime,payment_method,device_type,ip_country,order_total,customer_id,full_name,email,customer_segment,loyalty_tier,model_version,scored_at_utc,proba_fraud,risk_band_1_100,is_fraud_pred
0,146,2025-11-19 17:04:13,card,desktop,US,1016.82,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.348373,35,1
1,355,2025-10-11 01:13:27,card,desktop,US,1002.73,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.276265,28,1
2,777,2025-12-21 19:28:57,card,mobile,US,674.09,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.268259,27,1
3,1475,2025-12-16 16:46:18,card,mobile,US,828.21,2,Juan Flores,juanflores1@example.com,budget,none,v1,2026-04-02T02:55:03.053355+00:00,0.236346,24,1
4,1592,2025-10-11 08:39:43,card,mobile,US,938.31,2,Juan Flores,juanflores1@example.com,budget,none,v1,2026-04-02T02:55:03.053355+00:00,0.235001,24,1
5,4513,2025-11-13 14:17:37,paypal,mobile,US,991.18,96,Gabriela Taylor,gabrielataylor95@example.com,budget,none,v1,2026-04-02T02:55:03.053355+00:00,0.234409,24,1
6,433,2025-10-02 12:42:00,paypal,tablet,US,1011.69,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.232030,24,1
7,703,2025-08-02 02:16:22,card,tablet,US,1019.81,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.232030,24,1
8,1456,2025-08-03 12:07:30,crypto,desktop,US,1378.29,2,Juan Flores,juanflores1@example.com,budget,none,v1,2026-04-02T02:55:03.053355+00:00,0.228179,23,1
9,987,2025-10-11 08:44:51,card,desktop,US,1644.46,1,Patricia Diallo,patriciadiallo0@example.com,standard,silver,v1,2026-04-02T02:55:03.053355+00:00,0.222764,23,1
